In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import category_encoders as ce

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.preprocessing import LabelEncoder,StandardScaler,Normalizer
from sklearn.metrics import f1_score,r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
import lightgbm as lgb
#import xgboost as xgb

import seaborn as sns
import matplotlib.pyplot as plt


import warnings
warnings.filterwarnings('ignore')

In [ ]:
#data1 = pd.read_csv('../data/favorita-grocery-sales-forecasting/train.csv')

#data1['date'] = pd.to_datetime(data1['date'])

#data1.to_parquet('train_optimized.parquet', engine='pyarrow')

In [3]:



data = pd.read_parquet('../data/train_optimized.parquet')

In [7]:
data_last_year = data[data['date'] >= pd.Timestamp('2016-08-15')].copy()

In [ ]:
#x_test1 = pd.read_csv('../data/favorita-grocery-sales-forecasting/test.csv')
#x_test1['date'] = pd.to_datetime(x_test1['date'])
#x_test1.to_parquet('output_test_optimized.parquet', engine='pyarrow')


id                      int64
date           datetime64[us]
store_nbr               int64
item_nbr                int64
onpromotion              bool
dtype: object

In [4]:

x_test = pd.read_parquet('../data/output_test_optimized.parquet')

x_test.dtypes

id                      int64
date           datetime64[us]
store_nbr               int64
item_nbr                int64
onpromotion              bool
dtype: object

In [ ]:
#data_oil1 = pd.read_csv('../data/favorita-grocery-sales-forecasting/oil.csv')
#data_oil1['date'] = pd.to_datetime(data_oil1['date'])
#data_oil1.to_parquet('output_oil_optimized.parquet', engine='pyarrow')
#

In [5]:

data_oil = pd.read_parquet('../data/output_oil_optimized.parquet')


In [8]:

data_oil = data_oil.sort_values('date')

# Заполняем пропуски выходных дней
data_oil['dcoilwtico'] = data_oil['dcoilwtico'].ffill().bfill()

# Считаем фичи на маленьком датасете
data_oil['oil_ma_7'] = data_oil['dcoilwtico'].rolling(window=21, min_periods=1).mean()
data_oil['oil_std_7'] = data_oil['dcoilwtico'].rolling(window=21, min_periods=1).std()
data_oil['oil_lag_14'] = data_oil['dcoilwtico'].shift(28)


In [ ]:
#data_items1 = pd.read_csv('../data/favorita-grocery-sales-forecasting/items.csv')

#data_items1.to_parquet('output_items.parquet', engine='pyarrow')


In [6]:

data_items = pd.read_parquet('../data/output_items.parquet')


In [11]:

# 2. ПОДГОТОВКА (до слияния)
# Оптимизируем типы данных для экономии памяти
data_items['family'] = data_items['family'].astype('category')
# class - это просто ID категории (до ~4000), хватит int16
data_items['class'] = data_items['class'].astype('category')
# perishable - бинарный флаг (0 или 1), хватит int8
data_items['item_nbr'] = data_items['item_nbr'].astype('category')
data_items['perishable'] = data_items['perishable'].astype('int8')

# Сразу рассчитываем веса для Kaggle-метрики
data_items['item_weight'] = np.where(data_items['perishable'] == 1, 1.25, 1.0).astype('float32')



In [12]:

data_items.dtypes

item_nbr       category
family         category
class          category
perishable         int8
item_weight     float32
dtype: object

In [13]:
# 1. СКЛЕИВАЕМ (test прилипает к концу train)
all_data = pd.concat([data_last_year, x_test], ignore_index=True)

# Уменьшаем вес колонки продаж до 32-битного формата (экономит массу RAM)
all_data['unit_sales'] = all_data['unit_sales'].astype('float32')


# 2. СОРТИРУЕМ общую таблицу (теперь тестовые дни встанут ровно после тренировочных для каждого товара)
all_data = all_data.sort_values(by=['store_nbr', 'item_nbr', 'date'])

# Шаг А: Считаем базовый сдвиг ОДИН РАЗ (это очень быстрая операция)
all_data['lag_16'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(16)

# Также ужимаем вес новой колонки
all_data['lag_16'] = all_data['lag_16'].astype('float32')

# Шаг Б: Считаем скользящие средние поверх УЖЕ сдвинутой колонки
all_data['mean_2'] = all_data.groupby(['store_nbr', 'item_nbr'])['lag_16'].transform(lambda x: x.rolling(2).mean()).astype('float32')
all_data['mean_4'] = all_data.groupby(['store_nbr', 'item_nbr'])['lag_16'].transform(lambda x: x.rolling(4).mean()).astype('float32')
all_data['mean_6'] = all_data.groupby(['store_nbr', 'item_nbr'])['lag_16'].transform(lambda x: x.rolling(6).mean()).astype('float32')

all_data['std_2'] = all_data.groupby(['store_nbr', 'item_nbr'])['lag_16'].transform(lambda x: x.rolling(2).std()).astype('float32')
all_data['std_4'] = all_data.groupby(['store_nbr', 'item_nbr'])['lag_16'].transform(lambda x: x.rolling(4).std()).astype('float32')
all_data['std_6'] = all_data.groupby(['store_nbr', 'item_nbr'])['lag_16'].transform(lambda x: x.rolling(6).std()).astype('float32')

# Дополнительные лаги
all_data['lag_21'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(21).astype('float32')
all_data['lag_28'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(28).astype('float32')




In [14]:

all_data = all_data.merge(data_oil, on='date', how='left')

In [15]:
# 3. ОБЪЕДИНЕНИЕ
# Теперь приклеиваем уже готовую, легкую и обогащенную таблицу к основному датасету
all_data = all_data.merge(data_items, on='item_nbr', how='left')

In [16]:

# 4. РАЗДЕЛЯЕМ ОБРАТНО
# Тренировочные данные — те, где изначально были продажи
#train_features = all_data[all_data['unit_sales'].notna()].copy()

# Тестовые данные — те, которые мы должны предсказать
#test_after_split = all_data[all_data['unit_sales'].isna()].copy()

train_features = all_data[all_data['date'] < '2017-08-16'].copy()
test_after_split  = all_data[all_data['date'] >= '2017-08-16'].copy()
# 3. Удаляем строки с NaN, которые образовались в начале train из-за лагов и окон
# (В test_features мы NaN не удаляем, там должно остаться ровно столько строк, сколько ждет Kaggle!)
train_features = train_features.dropna()

In [11]:
print(train_features.dtypes)
test_after_split.dtypes


id                           int64
date                datetime64[us]
store_nbr                    int64
item_nbr                     int64
unit_sales                 float64
onpromotion                 object
unit_sales_lag_1           float64
unit_sales_lag_7           float64
dtype: object


id                           int64
date                datetime64[us]
store_nbr                    int64
item_nbr                     int64
unit_sales                 float64
onpromotion                 object
unit_sales_lag_1           float64
unit_sales_lag_7           float64
dtype: object

In [17]:

y = train_features.unit_sales
X = train_features.drop('unit_sales',axis=1)
#print(X['date'].max())



# Извлекаем полезные числовые признаки из даты
X['year'] = X['date'].dt.year
X['month'] = X['date'].dt.month
X['day'] = X['date'].dt.day

# День недели (от 0 до 6) — критически важный признак для прогнозирования продаж!
X['day_of_week'] = X['date'].dt.dayofweek 

for col in X.select_dtypes(include=['float64']).columns:
    X[col] = X[col].astype('float32')

for col in X.select_dtypes(include=['int64']).columns:
    X[col] = X[col].astype('int32')


In [18]:
test_ids = test_after_split['id']

# 2. Удаляем id из тестовой выборки (модель должна видеть только признаки!)
x_test_features = test_after_split.drop('id', axis=1)
x_test_features['date'] = pd.to_datetime(x_test_features['date'])

# Извлекаем полезные числовые признаки из даты
x_test_features['year'] = x_test_features['date'].dt.year
x_test_features['month'] = x_test_features['date'].dt.month
x_test_features['day'] = x_test_features['date'].dt.day

# День недели (от 0 до 6) — критически важный признак для прогнозирования продаж!
x_test_features['day_of_week'] = x_test_features['date'].dt.dayofweek 


# Удаляем оригинальный столбец с датой, чтобы он не ломал модель
x_test_features = x_test_features.drop('date', axis=1)

x_test_features['onpromotion'] = x_test_features['onpromotion'].map({True: 1, False: 0}).fillna(-1).astype(int)

x_test_features = x_test_features.drop(['unit_sales'], axis=1)

In [ ]:
# Удаляем оригинальный столбец с датой, чтобы он не ломал модель
X = X.drop('date', axis=1)

train_ids = X['id']

# 2. Удаляем id из тестовой выборки (модель должна видеть только признаки!)
x_train_features = X.drop('id', axis=1)

x_train_features['onpromotion'] = x_train_features['onpromotion'].map({True: 1, False: 0}).fillna(-1).astype(int)


KeyError: "['date'] not found in axis"

In [18]:

print(x_train_features.shape)
print(x_train_features.dtypes)
print(x_test_features.shape)
x_test_features.dtypes


(37454835, 9)
store_nbr             int32
item_nbr              int32
onpromotion           int64
unit_sales_lag_1    float32
unit_sales_lag_7    float32
year                  int32
month                 int32
day                   int32
day_of_week           int32
dtype: object
(3370464, 9)


store_nbr             int64
item_nbr              int64
onpromotion           int64
unit_sales_lag_1    float64
unit_sales_lag_7    float64
year                  int32
month                 int32
day                   int32
day_of_week           int32
dtype: object

In [25]:
x_train_features = X.drop('item_nbr', axis=1)
print(x_train_features.dtypes)

id                int32
store_nbr         int32
onpromotion      object
lag_16          float32
mean_2          float32
mean_4          float32
mean_6          float32
std_2           float32
std_4           float32
std_6           float32
lag_21          float32
lag_28          float32
dcoilwtico      float32
oil_ma_7        float32
oil_std_7       float32
oil_lag_14      float32
family         category
class          category
perishable         int8
item_weight     float32
year              int32
month             int32
day               int32
day_of_week       int32
dtype: object


In [24]:

#model = lgb.LGBMRegressor(learning_rate=0.1,n_estimators=100,max_depth=5,num_leaves=50)
model = lgb.LGBMRegressor(n_estimators=300,max_depth=10,random_state=42,n_jobs=-1)
model.fit(x_train_features,y)
predictions = model.predict(x_test_features)


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: onpromotion: object

In [ ]:
submission = pd.DataFrame({
    'id': test_ids,         # Подставляем сохраненные номера из пункта 1
    'unit_sales': predictions    # Подставляем результат из пункта 3
})
submission['unit_sales'] = submission['unit_sales'].clip(lower=0)
# 5. Сохраняем в файл строго без внутренних индексов Pandas
submission.to_csv('../data/submission9.csv', index=False)

date              str
dcoilwtico    float64
dtype: object

In [40]:
data_oil.describe()

,dcoilwtico
count,1175.000000
mean,67.714366
std,25.630476
min,26.190000
25%,46.405000
50%,53.190000
75%,95.660000
max,110.620000


In [42]:
print("Самая ранняя дата:", data_oil['date'].min())
print("Самая поздняя дата:", data_oil['date'].max())

Самая ранняя дата: 2013-01-01
Самая поздняя дата: 2017-08-31
